In [2]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("spam_train_cleaned.csv")
test_df = pd.read_csv("spam_test_cleaned.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

display(train_df.head())

Train shape: (31311, 16)
Test shape : (7828, 16)


,label,urls,hour,combined_text,capital_letter_count,capital_ratio,exclamation_count,question_count,special_char_count,day_of_week_Friday,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
0,0,1,2,volunteers are needed for alumni phonothon 200...,37,0.055306,0,0,2,0,0,0,0,0,0,1
1,0,0,0,re opensuse opensuse and faxes on sunday 10 fe...,85,0.048935,0,2,42,0,0,0,0,0,0,1
2,0,1,2,re r matching a period in grep on 08 05 2008 0...,69,0.045128,0,4,114,0,0,0,0,0,0,1
3,1,1,17,fast and safe male enhancement huge love gun i...,11,0.034700,3,0,0,0,0,0,0,1,0,0
4,0,1,3,re python dev documentation reorganization was...,44,0.026113,0,0,94,0,0,0,0,0,0,1


In [3]:
#Check duplicates
print("Train duplicates:", train_df["combined_text"].duplicated().sum())
print("Test duplicates:", test_df["combined_text"].duplicated().sum())

Train duplicates: 3900
Test duplicates: 793


In [4]:
# Remove duplicate emails from train and test
train_df = train_df.drop_duplicates(subset="combined_text").reset_index(drop=True)
test_df = test_df.drop_duplicates(subset="combined_text").reset_index(drop=True)

print("Train shape after removing duplicates:", train_df.shape)
print("Test shape after removing duplicates:", test_df.shape)

print("Train duplicates:", train_df["combined_text"].duplicated().sum())
print("Test duplicates:", test_df["combined_text"].duplicated().sum())

Train shape after removing duplicates: (27411, 16)
Test shape after removing duplicates: (7035, 16)
Train duplicates: 0
Test duplicates: 0


In [5]:
# check class balance 
print("Train class distribution:")
print(train_df["label"].value_counts())

print("\nTest class distribution:")
print(test_df["label"].value_counts())

Train class distribution:
label
0    13790
1    13621
Name: count, dtype: int64

Test class distribution:
label
1    3580
0    3455
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix


# X and y


X = train_df.drop(columns=["label"])
y = train_df["label"]

X_test_raw = test_df.drop(columns=["label"])
y_test = test_df["label"]



# Train / Validation split


X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)



# Text + other features


TEXT_COL = "combined_text"

NUMERIC_COLS = [
    col for col in X.columns
    if col != TEXT_COL
]



# TF-IDF for text


tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_text_train = tfidf.fit_transform(
    X_train_raw[TEXT_COL]
)

X_text_val = tfidf.transform(
    X_val_raw[TEXT_COL]
)

X_text_test = tfidf.transform(
    X_test_raw[TEXT_COL]
)



# Combine text + features


X_train = hstack([
    X_text_train,
    csr_matrix(X_train_raw[NUMERIC_COLS].values)
])

X_val = hstack([
    X_text_val,
    csr_matrix(X_val_raw[NUMERIC_COLS].values)
])

X_test = hstack([
    X_text_test,
    csr_matrix(X_test_raw[NUMERIC_COLS].values)
])


print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (21928, 314)
Validation: (5483, 314)
Test: (7035, 314)


In [7]:
# Defualt decision tree
from sklearn.tree import DecisionTreeClassifier

dt_default = DecisionTreeClassifier(
    random_state=42
)

dt_default.fit(X_train, y_train) # the training uses the full 39k not only 8k samples as in lazy classifer

print("Default Decision Tree trained!")

Default Decision Tree trained!


In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_train_pred_dt = dt_default.predict(X_train)
y_val_pred_dt = dt_default.predict(X_val)


print("========== DEFAULT DECISION TREE ==========")

print("\nTRAINING PERFORMANCE")
print("Accuracy :", round(accuracy_score(y_train, y_train_pred_dt), 4))
print("Precision:", round(precision_score(y_train, y_train_pred_dt), 4))
print("Recall   :", round(recall_score(y_train, y_train_pred_dt), 4))
print("F1 Score :", round(f1_score(y_train, y_train_pred_dt), 4))


print("\nVALIDATION PERFORMANCE")
print("Accuracy :", round(accuracy_score(y_val, y_val_pred_dt), 4))
print("Precision:", round(precision_score(y_val, y_val_pred_dt), 4))
print("Recall   :", round(recall_score(y_val, y_val_pred_dt), 4))
print("F1 Score :", round(f1_score(y_val, y_val_pred_dt), 4))


print("\nCONFUSION MATRIX")
print(confusion_matrix(y_val, y_val_pred_dt))


print("\nCLASSIFICATION REPORT")
print(classification_report(y_val, y_val_pred_dt))

========== DEFAULT DECISION TREE ==========

TRAINING PERFORMANCE
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

VALIDATION PERFORMANCE
Accuracy : 0.9777
Precision: 0.9734
Recall   : 0.982
F1 Score : 0.9777

CONFUSION MATRIX
[[2685   73]
 [  49 2676]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.98      0.97      0.98      2758
           1       0.97      0.98      0.98      2725

    accuracy                           0.98      5483
   macro avg       0.98      0.98      0.98      5483
weighted avg       0.98      0.98      0.98      5483



default Decision Tree is overfitting
:

Training F1 = 1.0000 (100%)
Validation F1 = 0.9777 (97.77%)
F1 gap = 2.23 percentage points

So unlike the SVM, the default tree is clearly overfitting.

The tree has learned the training data almost perfectly, but performance drops when it sees unseen validation data.
- the default setup 

<code>DecisionTreeClassifier(
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1
)</code>

In [10]:
# Experminet with diffrent tree depth


from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

depth_values = [5, 10, 15, 20, 30, None]

depth_results = []

for depth in depth_values:

    dt = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    # Train on the full training split
    dt.fit(X_train, y_train)

    # Predictions
    train_pred = dt.predict(X_train)
    val_pred = dt.predict(X_val)

    # Metrics
    train_accuracy = accuracy_score(y_train, train_pred)
    val_accuracy = accuracy_score(y_val, val_pred)

    train_f1 = f1_score(y_train, train_pred)
    val_f1 = f1_score(y_val, val_pred)

    # Store results
    depth_results.append({
        "Max Depth": depth,
        "Train Accuracy": train_accuracy,
        "Validation Accuracy": val_accuracy,
        "Train F1": train_f1,
        "Validation F1": val_f1,
        "F1 Gap": train_f1 - val_f1
    })


# Results table
depth_results_df = pd.DataFrame(depth_results)

display(depth_results_df.round(4))

,Max Depth,Train Accuracy,Validation Accuracy,Train F1,Validation F1,F1 Gap
0,5.0,0.9730,0.9690,0.9729,0.9688,0.0041
1,10.0,0.9908,0.9796,0.9908,0.9795,0.0112
2,15.0,0.9938,0.9796,0.9938,0.9795,0.0143
3,20.0,0.9956,0.9799,0.9956,0.9799,0.0157
4,30.0,0.9984,0.9788,0.9984,0.9788,0.0197
5,NaN,1.0000,0.9777,1.0000,0.9777,0.0223


As depth increases:Tree becomes more complex → training F1 increases → validation F1 stops improving → train/validation gap gets bigger.

- Validation F1 was used as the primary metric for selecting max_depth because it balances precision and recall. A depth of 20 achieved the highest validation F1 (97.99%). However, depth 10 produced nearly identical validation performance (97.95%) with a smaller train-validation gap, making it a more conservative choice if minimizing overfitting is prioritized.

- In summary: Selected max_depth = 10 because it provides nearly the best validation F1 while maintaining a smaller train-validation gap, resulting in better generalization with less overfitting